In [1]:
import os
from databricks.sdk import WorkspaceClient
from dotenv import load_dotenv

# PRE-REQUISITES:
# generate the client ID and secret for the service principal separately in Databricks and set them in the traffic_agent.env file
# grant the SP CAN_QUERY permission on the endpoint
load_dotenv(dotenv_path="traffic_agent.env", override=True)

w = WorkspaceClient(
    host="https://dbc-564fb500-5a75.cloud.databricks.com/",
    client_id=os.environ.get("DATABRICKS_CLIENT_ID"),
    client_secret=os.environ.get("DATABRICKS_CLIENT_SECRET")
)

In [3]:
import os
import requests
import pandas as pd
import json


def create_tf_serving_json(query: str):
    """
    Add the mlflow required "input" wrapper
    """
    return {"inputs": [{"accident_texts": [{"accident_text": query}]}]}


def get_oauth_token(databricks_host, client_id, client_secret):
    """
    Get OAuth token using service principal credentials

    Args:
        databricks_host: Your Databricks workspace URL (e.g., "https://dbc-564fb500-5a75.cloud.databricks.com")
        client_id: Service principal client ID
        client_secret: Service principal client secret

    Returns:
        dict: Token response containing access_token, token_type, expires_in
    """
    # Construct the token endpoint URL
    token_url = f"{databricks_host}/oidc/v1/token"

    # Prepare the request
    headers = {"Content-Type": "application/x-www-form-urlencoded"}

    # Method 1: Using Basic Authentication (recommended)
    auth = (client_id, client_secret)
    data = {
        "grant_type": "client_credentials",
        "scope": "all-apis",  # or specify specific scopes
    }

    response = requests.post(url=token_url, headers=headers, auth=auth, data=data)

    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(
            f"Token request failed: {response.status_code} - {response.text}"
        )


def score_model(query: str, databricks_host, endpoint_name, client_id, client_secret):
    """
    Score a model deployed to a Databricks Serving Endpoint.

    Args:
        query: Input query string (e.g., "what is the volume of a cylinder with height of 5 and radius of 2?")
        databricks_host: Your Databricks workspace URL (e.g., "https://dbc-564fb500-5a75.cloud.databricks.com")
        endpoint_name: Name of the serving endpoint
        client_id: Service principal client ID
        client_secret: Service principal client secret

    Returns:
        dict: Model prediction response

    """

    url = f"{databricks_host}/serving-endpoints/{endpoint_name}/invocations"
    token_response = get_oauth_token(databricks_host, client_id, client_secret)
    access_token = token_response["access_token"]

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }

    ds_dict = create_tf_serving_json(query)

    data_json = json.dumps(ds_dict, allow_nan=True)
    response = requests.request(method="POST", headers=headers, url=url, data=data_json)
    if response.status_code != 200:
        raise Exception(
            f"Request failed with status {response.status_code}, {response.text}"
        )
    return response.json()

In [6]:
query = """
On 2026-03-15 at approximately 09:45, a Honda Civic (plate number DEF789) rear-ended a Ford F-150 pickup truck (plate number LMN012) on Highway 405. 
The accident resulted in moderate damage to both vehicles and minor whiplash injuries to the driver of the Honda Civic. 
Emergency services arrived within 10 minutes, and no fatalities were reported."""

response = score_model(
    query,
    databricks_host="https://dbc-564fb500-5a75.cloud.databricks.com",
    endpoint_name="road_traffic_accident_analysis_agent-endpoint",
    client_id=os.environ.get("DATABRICKS_CLIENT_ID"),
    client_secret=os.environ.get("DATABRICKS_CLIENT_SECRET")
)
response

{'predictions': [{'results': [{'result': {'input_text': '\nOn 2026-03-15 at approximately 09:45, a Honda Civic (plate number DEF789) rear-ended a Ford F-150 pickup truck (plate number LMN012) on Highway 405. \nThe accident resulted in moderate damage to both vehicles and minor whiplash injuries to the driver of the Honda Civic. \nEmergency services arrived within 10 minutes, and no fatalities were reported.',
      'accident_overview': [{'title': 'Rear-end collision on Highway 405',
        'date': '2026-03-15',
        'time': '09:45',
        'reported_injury': 'yes',
        'reported_fatality': 'no',
        'span_id': None}],
      'vehicle_analysis': [{'accident_vehicle': [{'vehicle_id': 'DEF789',
          'vehicle_type': 'Passenger Car'},
         {'vehicle_id': 'LMN012', 'vehicle_type': 'Truck'}],
        'span_id': None}],
      'location_analysis': [{'accident_location': [{'location_name': 'Highway 405',
          'location_type': 'Expressways/Freeways'}],
        'span_id':